# Vue2 + Vue3 框架 · 第 09 天

## 智慧商城项目：首页、搜索、商品详情与加入购物车

> 本篇整理项目的主要浏览链路：首页数据、搜索历史和结果、商品详情、规格选择、数量框、登录确认和加入购物车。

## 一、首页动态渲染

首页通常包含轮播图、分类入口、猜你喜欢和推荐商品。页面进入时在 `created` 请求数据，模板只负责渲染：

```js
import { getHomeData } from '@/api/home'
export default {
  data () { return { banner: [], category: [], goods: [] } },
  async created () {
    const { data } = await getHomeData()
    this.banner = data.banner
    this.category = data.category
    this.goods = data.goods
  }
}
```

Vant 轮播示例：

```vue
<van-swipe :autoplay="3000">
  <van-swipe-item v-for="item in banner" :key="item.imgUrl">
    <img :src="item.imgUrl" alt="">
  </van-swipe-item>
</van-swipe>
```

把接口字段转换成页面需要的结构，可以在 API 层或 computed 中完成，避免模板中出现大量字段判断。

## 二、搜索页与历史记录

### 2.1 搜索关键词

```js
data () {
  return { searchValue: '', history: JSON.parse(localStorage.getItem('search-history') || '[]'), list: [], page: 1 }
},
methods: {
  submitSearch () {
    const words = this.searchValue.trim()
    if (!words) return
    this.history = [words, ...this.history.filter(item => item !== words)].slice(0, 10)
    localStorage.setItem('search-history', JSON.stringify(this.history))
    this.$router.push({ path: '/search/list', query: { search: words } })
  },
  clearHistory () {
    this.history = []
    localStorage.removeItem('search-history')
  }
}
```

搜索历史需要去重、限制数量，并在页面初始化时从本地存储恢复。点击历史项可以直接把关键词写回输入框并提交。

### 2.2 搜索结果

```js
async getList () {
  const { data } = await getSearchList({ search: this.$route.query.search, page: this.page })
  this.list = data.list
}
```

关键词变化或分页变化时重新请求；空结果显示空状态，加载和错误状态要有反馈。

## 三、分类 id 与商品列表

搜索可以按关键字或分类 id：

```js
this.$router.push({
  path: '/search/list',
  query: { categoryId: item.categoryId }
})
```

页面根据 `this.$route.query.search` 或 `categoryId` 组织请求参数。不要把两个参数写死在组件里，建议通过计算属性统一生成：

```js
computed: {
  queryParams () {
    return {
      search: this.$route.query.search || '',
      categoryId: this.$route.query.categoryId || '',
      page: this.page
    }
  }
}
```

## 四、商品详情与 SKU

详情页通过路由参数获取商品 id：

```js
computed: {
  goodsId () { return this.$route.params.id || this.$route.query.goodsId }
},
async created () {
  const { data } = await getGoodsDetail(this.goodsId)
  this.detail = data
}
```

详情接口通常包含主图、价格、库存、规格列表、商品介绍和评价。规格选择应维护一个 `selectedSku`，当规格变化时重新计算当前 SKU 的库存和价格。

```js
computed: {
  selectedSku () {
    return this.detail.skuList.find(item => item.goods_sku_id === this.selectedSkuId) || {}
  }
}
```

渲染富文本时注意接口内容的可信度，商品图片用 `:src` 绑定并提供默认图。

## 五、数字框组件与加入购物车面板

数量框组件应通过 props 接收最小值、最大值和当前值，通过事件通知父组件：

```vue
<template>
  <div class="input-number">
    <button :disabled="value <= min" @click="change(-1)">-</button>
    <span>{{ value }}</span>
    <button :disabled="value >= max" @click="change(1)">+</button>
  </div>
</template>
<script>
export default {
  props: { value: Number, min: { type: Number, default: 1 }, max: { type: Number, default: 99 } },
  methods: {
    change (step) {
      const next = Math.min(this.max, Math.max(this.min, this.value + step))
      this.$emit('input', next)
    }
  }
}
</script>
```

商品详情点击“加入购物车”时打开规格面板，用户选择 SKU 和数量后才提交请求。组件的职责是展示和收集选择，接口调用放在页面方法中。

## 六、登录确认与 mixin 复用

加入购物车、立即购买等操作都需要登录，可以提取为 mixin：

```js
// mixins/loginConfirm.js
export default {
  methods: {
    loginConfirm () {
      if (this.$store.getters['user/token']) return false
      this.$dialog.confirm({
        title: '温馨提示',
        message: '请先登录后继续操作',
        confirmButtonText: '去登录'
      }).then(() => {
        this.$router.push({ path: '/login', query: { backUrl: this.$route.fullPath } })
      }).catch(() => {})
      return true
    }
  }
}
```

页面中：

```js
import loginConfirm from '@/mixins/loginConfirm'
export default { mixins: [loginConfirm] }
```

```js
async addCart () {
  if (this.loginConfirm()) return
  await addCart(this.goodsId, this.addCount, this.selectedSkuId)
  this.$toast.success('加入购物车成功')
}
```

mixin 适合小范围复用方法；逻辑复杂时可以抽成独立工具或组合式函数，避免多个 mixin 之间命名冲突。

## 七、API 层与页面状态

建议按业务拆分接口：

```js
// api/goods.js
export const getHomeData = () => request.get('/home/index')
export const getSearchList = params => request.get('/goods/list', { params })
export const getGoodsDetail = goodsId => request.get(`/goods/detail/${goodsId}`)
export const addCart = (goodsId, goodsNum, goodsSkuId) => request.post('/cart/add', { goodsId, goodsNum, goodsSkuId })
```

页面状态至少区分：

- `loading`：请求中。
- `error`：请求失败。
- `empty`：成功但没有数据。
- `data`：真正用于渲染的数据。

避免在模板中直接写长串接口字段和业务判断，让组件更容易阅读和测试。

## 八、当天速记与练习

- 首页、搜索、详情页都遵循“路由参数 → 请求 → 响应式数据 → 模板渲染”。
- 搜索历史用数组去重、限制长度并持久化到 `localStorage`。
- 详情页把规格选择和数量选择放在独立组件中，通过 `v-model` 或事件同步。
- 加入购物车前先检查 token，登录后通过 `backUrl` 回到原页面。
- API 请求集中封装，页面只负责组合参数和处理业务状态。

练习：完成商品详情到购物车的完整链路，分别测试未登录、无库存、切换规格和加入成功四种情况。